# B3PN — Part-mask filter (P) and normals merge detector (N)

## Why
B3MX: **MixA** (zoomed automatic masks from Z-LM0 + unzoomed guided masks from LM0) gives the first **significant** overall gains over H1b_v1 (GPV_K-n[MixA] 26.11, −2.54; H1b[MixA] 26.62, −2.03) but fails T1. Its extra S1 error has two separable sources:
- **+2.6** from zoomed automatic **part-masks** (GPV_R4-L[M0] S1 5.35 → GPV_R4[MixA] 7.98);
- **+5.0 more** from **false merge flags** in GPV_K-n (GPV_R4[MixA] 7.98 → GPV_K-n[MixA] 13.02): rule A and Gate 1 rely on coarse D1 semantic peaks, so single objects get split by K3n.

Normals already showed that a clean boundary signal fixes false splitting (GPV_K S3 26.87 → 13.70). Here they also decide **whether** a mask is merged.

## Pre-declared design (before any B3PN result)
| Fix | Rule |
|---|---|
| **Object size s** | median area of the image's verified single masks (target-dominant in the tiled expanded-vocabulary map, not rule-A merged, SAM2 score ≥ image median); fallbacks: target-dominant masks, then all masks (UM definition, full mask area) |
| **P — part-mask filter** | drop a mask if its area < **0.25 s** (a quarter of an object — clearly a part) **and** > **80%** of it lies inside another kept mask (nested) |
| **N — normals merge detector** (GPV_K-n) | a mask is flagged as merged only if its area > 4 × the median mask area (rule A's size condition) **and** K3n finds **≥ 2 segments** inside it (min segment 0.25 × median mask area); Gate 1's D1-peak sum is replaced by the **K3n segment sum** over qualifying regions (> number of unflagged masks); counting otherwise as GPV_K-n |
| Bases | **MixA** and **LM0** |
| Variants | GPV_R4, GPV_K-n, GPV_R4+P, GPV_K-n+P, GPV_K-n+N, GPV_K-n+P+N, H1b-K3n, H1b-K3n+P |
| Targets | T1, T2′, T3′ |

## Run order
CPU → 1.0–1.3 → 2.1 → 3.1 (resumable) → 4.1.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3PN_Part_and_Normals_Merge"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3PN_Part_and_Normals_Merge


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


In [4]:
# ============================================================
# Step 1.3 — Normals edge map and K3n counting (B3MN definitions)
# ============================================================

from scipy import ndimage
import src.pipeline.multiplicity as K
importlib.reload(K)
N_ROOT = TD_ROOT / "normals"

def normal_edge_map(nrm):
    n = nrm.astype(np.float32)
    n = n / np.maximum(np.linalg.norm(n, axis=2, keepdims=True), 1e-6)
    edge = np.zeros(n.shape[:2], np.float32)
    for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        edge = np.maximum(edge, 1.0 - (n * np.roll(n, shift=(dy, dx), axis=(0, 1))).sum(axis=2))
    edge[0, :] = edge[1, :]; edge[-1, :] = edge[-2, :]; edge[:, 0] = edge[:, 1]; edge[:, -1] = edge[:, -2]
    return ndimage.gaussian_filter(edge, 1.0)

def k3n_segments(region, edge, min_seg, h_frac=0.1):
    e = edge[region]
    h = h_frac * float(np.percentile(e, 95) - np.percentile(e, 5))
    return 1 if h <= 0 else K._marker_watershed(edge, region, -edge, h, min_seg)

def gpv_k_n_count(r4_m, r4_s, d1, edge):
    masks = np.asarray(r4_m, bool)
    if masks.shape[0] == 0:
        return 0
    flagged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, r4_s, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0])
    return n_nonflag + sum(max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions)

def h1b_k3n_count(r4_m, d1, a3c, depth, edge):
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = r4_m[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            total += max(1, k3n_segments(rg, edge, 0.25 * a_ref))
    return float(total)

def all_counts(masks, scores, d1, a3c, depth, edge):
    return {"GPV_R4": int(masks.shape[0]),
            "H1b": float(hybrid.h1b_count(masks, d1, a3c, depth, SG_CFG)[0]),
            "GPV_K-n": gpv_k_n_count(masks, scores, d1, edge),
            "H1b-K3n": h1b_k3n_count(masks, d1, a3c, depth, edge)}
print("K3n counting ready")

K3n counting ready


# 2. Fixes P and N

In [5]:
# ============================================================
# Step 2.1 — Object size, part-mask filter (P), normals merge detector (N)
# ============================================================

TD_TILED = TD_ROOT / "san_vocab_tiled"
PART_FRAC, NEST_FRAC = 0.25, 0.80

def object_size(masks, scores, amax, d1):
    n = masks.shape[0]
    areas = masks.reshape(n, -1).sum(1).astype(float)
    n_cls = int(amax.max()) + 1
    dom_t = np.array([np.bincount(amax[m], minlength=n_cls).argmax() == 0 if m.any() else False for m in masks])
    merged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    verified = dom_t & ~merged & (np.asarray(scores) >= np.median(scores))
    if verified.sum() >= 3:
        return float(np.median(areas[verified]))
    if dom_t.any():
        return float(np.median(areas[dom_t]))
    return float(np.median(areas))

def part_filter(masks, scores, amax, d1):
    n = masks.shape[0]
    if n < 2:
        return masks, np.asarray(scores, float), 0
    s = object_size(masks, scores, amax, d1)
    flat = masks.reshape(n, -1).astype(np.float32)
    areas = flat.sum(1)
    inter = flat @ flat.T
    np.fill_diagonal(inter, 0)
    nested = (inter.max(axis=1) > NEST_FRAC * np.maximum(areas, 1))
    drop = (areas < PART_FRAC * s) & nested
    keep = ~drop
    return masks[keep], np.asarray(scores, float)[keep], int(drop.sum())

def gpv_k_n_count_N(masks, scores, edge):
    masks = np.asarray(masks, bool)
    n = masks.shape[0]
    if n == 0:
        return 0, 0
    areas = masks.reshape(n, -1).sum(1).astype(float)
    med = float(np.median(areas))
    cand = np.flatnonzero(areas > 4.0 * med)
    flagged = np.zeros(n, bool)
    for i in cand:
        if k3n_segments(masks[i], edge, 0.25 * med) >= 2:
            flagged[i] = True
    if not flagged.any():
        return n, 0
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, scores, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    segs = [max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions]
    if not (regions and sum(segs) > n_nonflag):
        return n, int(flagged.sum())
    return n_nonflag + sum(segs), int(flagged.sum())
print("Fixes P and N ready")

Fixes P and N ready


# 3. Apply to the MixA and LM0 bases

In [6]:
# ============================================================
# Step 3.1 — All variants on MixA and LM0 (resumable per image)
# ============================================================

LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
ZLM_ROOT = TD_ROOT / "zoomed" / "LM0"
ROWS = OUT_DIR / "rows"
ROWS.mkdir(parents=True, exist_ok=True)

def r4_from(b0_dir, guided_dir, di, stem):
    bm, bmeta = io_cache.load_b0_masks(b0_dir / "b0" / f"{int(di):05d}_{stem}.npz")
    g = io_cache.load_guided_masks(guided_dir / "guided" / f"{int(di):04d}_{stem}.npz")
    kg = gpv.nms_masks(g["masks"], g["scores"], 0.5) if g["masks"].shape[0] else np.zeros(0, int)
    m, s, _ = gpv.build_r4(g["masks"][kg], np.asarray(g["scores"])[kg], bm, bmeta["predicted_iou"], 0.5)
    return m, np.asarray(s, float)

BASES = {"MixA": (ZLM_ROOT, LM_ROOT), "LM0": (LM_ROOT, LM_ROOT)}
todo = [r for r in td_df.itertuples(index=False) if not (ROWS / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="P / N variants"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    with np.load(N_ROOT / f"{stem}.npz") as v:
        edge = normal_edge_map(v["normals"])
    with np.load(TD_TILED / f"{stem}.npz") as v:
        amax = v["tiled_argmax"]
    row = {"dataset_index": int(di)}
    for b, (bdir, gdir) in BASES.items():
        m, s = r4_from(bdir, gdir, di, stem)
        mp, sp, n_drop = part_filter(m, s, amax, d1)
        kN, fN = gpv_k_n_count_N(m, s, edge)
        kPN, fPN = gpv_k_n_count_N(mp, sp, edge)
        row.update({
            f"GPV_R4[{b}]": int(m.shape[0]),
            f"GPV_K-n[{b}]": gpv_k_n_count(m, s, d1, edge),
            f"GPV_R4+P[{b}]": int(mp.shape[0]),
            f"GPV_K-n+P[{b}]": gpv_k_n_count(mp, sp, d1, edge),
            f"GPV_K-n+N[{b}]": kN,
            f"GPV_K-n+P+N[{b}]": kPN,
            f"H1b-K3n[{b}]": h1b_k3n_count(m, d1, a3c, dep, edge),
            f"H1b-K3n+P[{b}]": h1b_k3n_count(mp, d1, a3c, dep, edge),
            f"parts_dropped[{b}]": n_drop,
            f"flagged_ruleA[{b}]": int(gpv.merged_mask_flags(m, gpv.points_per_mask(m, d1), "A").sum()) if m.shape[0] else 0,
            f"flagged_N[{b}]": fN})
    with open(ROWS / f"{int(di):05d}.json", "w") as f:
        json.dump(row, f)
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


P / N variants:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 59.7 min


# 4. Results and targets

In [7]:
# ============================================================
# Step 4.1 — Diagnostics, metrics and targets
# ============================================================

ROWS = OUT_DIR / "rows"
pn = pd.DataFrame([json.load(open(p)) for p in sorted(ROWS.glob("*.json"))])
assert len(pn) == 400, len(pn)
frame = pn.merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                 .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")

print("Parts dropped and merge flags (rule A vs normals) per image, by stratum:")
dc = [f"{k}[{b}]" for b in ["MixA", "LM0"] for k in ["parts_dropped", "flagged_ruleA", "flagged_N"]]
display(frame.groupby("density_stratum")[dc].mean().reindex(STRATA).round(2))

VARS = ["GPV_R4", "GPV_K-n", "GPV_R4+P", "GPV_K-n+P", "GPV_K-n+N", "GPV_K-n+P+N", "H1b-K3n", "H1b-K3n+P"]
SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1"] + [f"{v}[{b}]" for b in ["MixA", "LM0"] for v in VARS]
res, targets = evaluate(frame, SYS, "Part-mask filter and normals merge detector (train_dev400)")

frame.to_csv(OUT_DIR / "B3PN_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3PN_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3PN_targets.csv", index=False)
with open(OUT_DIR / "B3PN_metadata.json", "w") as f:
    json.dump({"notebook": "B3PN_Part_and_Normals_Merge",
               "object_size": "median area of verified singles (target-dominant tiled vocab, not rule-A merged, "
                              "score >= image median); fallbacks target-dominant, all",
               "P": f"drop mask if area < {PART_FRAC} x object size AND > {NEST_FRAC:.0%} inside another kept mask",
               "N": "flag if area > 4 x median AND K3n >= 2 segments (min seg 0.25 x median); "
                    "Gate 1 uses K3n segment sum over qualifying regions > unflagged masks",
               "bases": {"MixA": "B0 masks Z-LM0 + guided LM0", "LM0": "SAM2.1-Large, masked input M0"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Parts dropped and merge flags (rule A vs normals) per image, by stratum:


,parts_dropped[MixA],flagged_ruleA[MixA],flagged_N[MixA],parts_dropped[LM0],flagged_ruleA[LM0],flagged_N[LM0]
density_stratum,,,,,,
S1_0_20,1.71,1.65,1.52,1.02,0.80,0.86
S2_20_40,1.94,1.00,0.99,1.06,0.76,0.78
S3_40_60,2.72,1.93,1.87,2.23,0.92,0.98
S4_60_80,3.05,2.25,1.43,1.35,1.28,1.02
S5_80_95,2.20,1.93,2.18,1.38,1.78,1.97
S6_95_100,2.62,2.47,3.37,0.82,2.00,2.55


Part-mask filter and normals merge detector (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_R4[MixA],27.02,-39.64,7.98,7.29,12.12,15.67,37.47,255.92
4,GPV_K-n[MixA],26.11,-32.99,13.02,8.22,12.18,16.02,29.88,234.73
5,GPV_R4+P[MixA],25.54,-41.96,6.26,5.40,9.73,13.25,37.93,258.37
6,GPV_K-n+P[MixA],24.38,-35.73,10.68,5.86,9.18,13.72,30.88,237.18
7,GPV_K-n+N[MixA],27.61,-24.71,14.54,9.28,15.92,21.07,27.30,227.05
8,GPV_K-n+P+N[MixA],25.69,-27.64,11.45,7.14,12.67,18.65,28.40,228.98
9,H1b-K3n[MixA],26.99,-30.78,9.78,8.39,14.27,17.30,32.78,242.47


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_R4[MixA],"+2.41 [+0.54, +4.40]","-0.45 [-2.44, +1.64]","+0.13 [-7.00, +8.73]","-19.07 [-36.48, -0.63]","-1.62 [-3.31, +0.15]",False,True,False,False
1,GPV_K-n[MixA],"+7.46 [+3.78, +11.47]","+0.49 [-1.89, +3.08]","-7.45 [-16.20, +2.42]","-40.25 [-65.99, -15.67]","-2.54 [-4.76, -0.26]",False,True,True,False
2,GPV_R4+P[MixA],"+0.70 [-0.99, +2.43]","-2.34 [-4.06, -0.46]","+0.60 [-6.48, +9.23]","-16.62 [-33.90, +1.67]","-3.11 [-4.81, -1.36]",True,True,True,True
3,GPV_K-n+P[MixA],"+5.11 [+1.89, +8.65]","-1.88 [-3.79, +0.16]","-6.45 [-15.05, +3.23]","-37.80 [-63.62, -13.47]","-4.27 [-6.40, -2.06]",False,True,True,False
4,GPV_K-n+N[MixA],"+8.97 [+4.24, +14.51]","+1.54 [-1.91, +6.16]","-10.03 [-17.97, -1.47]","-47.93 [-75.75, -21.48]","-1.04 [-3.75, +1.84]",False,True,False,False
5,GPV_K-n+P+N[MixA],"+5.89 [+1.70, +11.10]","-0.60 [-3.67, +3.82]","-8.93 [-16.57, -0.60]","-46.00 [-74.02, -19.57]","-2.96 [-5.65, -0.15]",False,True,True,False
6,H1b-K3n[MixA],"+4.21 [+1.92, +6.63]","+0.65 [-1.66, +3.07]","-4.55 [-10.83, +2.82]","-32.52 [-53.28, -10.93]","-1.66 [-3.53, +0.32]",False,True,False,False
7,H1b-K3n+P[MixA],"+2.53 [+0.40, +4.68]","-1.27 [-3.25, +0.80]","-4.02 [-9.97, +3.12]","-30.70 [-51.47, -9.27]","-3.25 [-5.10, -1.31]",False,True,True,False
8,GPV_R4[LM0],"-0.21 [-1.59, +1.11]","-2.46 [-3.99, -0.89]","+9.95 [+3.22, +18.33]","+42.72 [+29.13, +57.53]","+1.71 [+0.17, +3.37]",True,False,False,False
9,GPV_K-n[LM0],"+2.75 [+0.25, +5.49]","-2.11 [-3.86, -0.24]","+2.47 [-6.15, +12.07]","-1.20 [-28.62, +23.70]","-0.94 [-3.05, +1.25]",False,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3PN_Part_and_Normals_Merge
